In [1]:
import pickle

with open('S2.pkl', 'rb') as f:
    data = pickle.load(f, encoding='latin1')

print(type(data))

print(data.keys())
print(data['signal']['chest'].keys())
print(data['signal']['wrist'].keys())


/tmp/ipykernel_1754/2318298600.py:4: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  data = pickle.load(f, encoding='latin1')


<class 'dict'>
dict_keys(['signal', 'label', 'subject'])
dict_keys(['ACC', 'ECG', 'EMG', 'EDA', 'Temp', 'Resp'])
dict_keys(['ACC', 'BVP', 'EDA', 'TEMP'])


In [2]:
c_acc = data['signal']['chest']['ACC']
c_ecg = data['signal']['chest']['ECG']
c_emg = data['signal']['chest']['EMG']
c_eda = data['signal']['chest']['EDA']
c_temp = data['signal']['chest']['Temp']
c_resp = data['signal']['chest']['Resp']

In [3]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

labels = data['label']
Y = np.where(labels == 2, 1, 0)
X = np.concatenate([c_acc, c_ecg, c_eda, c_emg, c_resp, c_temp], axis=1)

stress_start = np.where(labels == 2)[0][0]
start_idx = max(0, stress_start - 5000)
end_idx = min(len(X), stress_start + 5000)
X = X[start_idx:end_idx]
Y = Y[start_idx:end_idx]

stress_idx = np.where(Y == 1)[0]
baseline_idx = np.where(Y == 0)[0]

idx = np.concatenate([stress_idx, baseline_idx])
np.random.seed(42)
np.random.shuffle(idx)

x_slice = X[idx]
y_slice = Y[idx]

split = int(len(x_slice) * 0.8)
X_train = x_slice[:split]
X_test = x_slice[split:]
y_train = y_slice[:split]
y_test = y_slice[split:]

scalar = StandardScaler()
X_train = scalar.fit_transform(X_train)
X_test = scalar.transform(X_test)

smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)
print("Decision Tree Classification Report:")
print(classification_report(y_test, dt_pred, target_names=['baseline', 'stress']))

lda = LinearDiscriminantAnalysis()
lda.fit(X_train, y_train)
y_pred_lda = lda.predict(X_test)
print("LDA Classification Report:")
print(classification_report(y_test, y_pred_lda, target_names=['baseline', 'stress']))

xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=(np.sum(y_train==0) / np.sum(y_train==1))
)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
print("XGBoost Classification Report:")
print(classification_report(y_test, y_pred_xgb, target_names=['baseline', 'stress']))

log_reg = LogisticRegression()
log_reg.fit(X_train, y_train)
y_pred_log = log_reg.predict(X_test)
print("Logistic Regression Classification Report:")
print(classification_report(y_test, y_pred_log, target_names=['baseline', 'stress']))



Decision Tree Classification Report:
              precision    recall  f1-score   support

    baseline       0.97      0.96      0.97      1020
      stress       0.96      0.97      0.97       980

    accuracy                           0.97      2000
   macro avg       0.97      0.97      0.97      2000
weighted avg       0.97      0.97      0.97      2000

LDA Classification Report:
              precision    recall  f1-score   support

    baseline       0.81      0.88      0.84      1020
      stress       0.86      0.79      0.82       980

    accuracy                           0.83      2000
   macro avg       0.84      0.83      0.83      2000
weighted avg       0.84      0.83      0.83      2000

XGBoost Classification Report:
              precision    recall  f1-score   support

    baseline       0.99      0.98      0.98      1020
      stress       0.97      0.99      0.98       980

    accuracy                           0.98      2000
   macro avg       0.98      0.98

Tried to do binary classification like in the paper
Joined 0,1,3,4.5,6,7 into the new unstressed condition
2 is the new stressed condition
reassigned unstressed condition as 0 and stressed condition as 1


Code wont run properly without SMOTE (too much bias towards unstressed condition)
First "Stressed" condition comes very late in the data so I need to change where its gathered from
Scalar for the large range of signals recorded